# 02 — Preprocessing

Cleans and resamples the raw water-level series.

**Inputs:** `data/raw/*.parquet`
**Outputs:** `data/interim/*.parquet`

In [ ]:
## Setup

from pathlib import Path

from tqdm.auto import tqdm

from src.config import MAX_INTERPOLATION_GAP_HOURS, STATION_IDS, WEATHER_VARIABLES
from src.fetch_data import summarize_failures
from src.preprocess import preprocess_station

RAW_DIR = Path("data/raw")
INTERIM_DIR = Path("data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
failures = {}

## Per-station hourly water level + weather

Each station's water-level history is reindexed to a strict hourly UTC grid;
gaps of at most `MAX_INTERPOLATION_GAP_HOURS` are linearly interpolated and
flagged via `imputed`, longer gaps stay `NaN`. GeoSphere INCA weather is left-joined
onto that grid, since the water-level timeline is the forecasting target.

In [ ]:
latest_station = None

for station_id in tqdm(STATION_IDS, desc="Preprocessing", unit="station"):
    interim_path = INTERIM_DIR / f"{station_id}_hourly.parquet"

    try:
        station = preprocess_station(
            station_id,
            raw_dir=RAW_DIR,
            max_gap_hours=MAX_INTERPOLATION_GAP_HOURS,
            weather_variables=WEATHER_VARIABLES,
        )
        station.to_parquet(interim_path, index=False)
        print(f"Saved {len(station):,} hourly rows to {interim_path}")
        latest_station = station
    except Exception as error:
        failures[str(interim_path)] = error
        print(f"Failed {interim_path}: {error}")

if latest_station is not None:
    latest_station.info()
    display(latest_station.head())
else:
    print("No preprocessed DataFrame is available to display.")

In [ ]:
if failures:
    raise RuntimeError(summarize_failures(failures))

print("Preprocessing completed successfully.")